# Milvus 本地学习笔记：下载、连接工具、PyMilvus 与 LangChain

本文档整理 Milvus 当前官方推荐的本地入门方式：先用 **Milvus Lite** 在本地文件中运行，再切换到 Docker Standalone 服务，随后使用 WebUI / Attu 连接管理，最后接入 LangChain 做向量检索和 RAG。

参考资料：

- Milvus Quickstart with Milvus Lite: https://milvus.io/docs/quickstart.md
- Milvus Docker Compose: https://milvus.io/docs/install_standalone-docker-compose.md
- Milvus LangChain Vector Store: https://milvus.io/docs/basic_usage_langchain.md
- LangChain Milvus integration: https://docs.langchain.com/oss/python/integrations/vectorstores/milvus
- LangChain Milvus API reference: https://reference.langchain.com/python/langchain-milvus/vectorstores/milvus/Milvus

> 安全说明：本 notebook 只演示创建新集合、插入测试数据、查询和搜索。示例不会删除、清空、重置已有 Milvus 数据。集合名会自动带时间戳，避免覆盖历史集合。

## 1. 环境准备

Milvus Lite 是 `pymilvus` 中包含的本地嵌入式向量数据库，适合 notebook、原型验证、小数据量学习。

推荐安装：

```bash
pip install -U pymilvus
pip install -U "pymilvus[model]"
pip install -U langchain-milvus langchain-openai langchain-core python-dotenv
```

如果使用 `uv`：

```bash
uv add pymilvus "pymilvus[model]" langchain-milvus langchain-openai langchain-core python-dotenv
```

说明：

- `pymilvus`：Milvus Python SDK，也包含 Milvus Lite。
- `pymilvus[model]`：包含官方示例使用的 embedding 工具，首次安装可能会拉取 PyTorch 等依赖。
- `langchain-milvus`：LangChain 官方 Milvus 集成包。
- `langchain-openai`：示例中用于 OpenAI Embeddings 和 Chat Model。也可以替换成你自己的 Embedding 模型。

In [ ]:
# 在 notebook 中安装依赖时取消下一行注释。
# %pip install -U pymilvus "pymilvus[model]" langchain-milvus langchain-openai langchain-core python-dotenv

## 2. Milvus 的两种本地运行方式

### 方式 A：Milvus Lite，本地文件版

最适合学习和 notebook。所有数据保存在一个本地 `.db` 文件里，不需要 Docker，不需要单独启动服务。

```python
from pymilvus import MilvusClient

client = MilvusClient("./milvus_demo.db")
```

### 方式 B：Docker Standalone，本地服务版

适合模拟真实服务环境。默认服务地址通常是：

- Milvus gRPC / HTTP endpoint: `http://localhost:19530`
- Milvus WebUI: `http://127.0.0.1:9091/webui/`

官方 Docker Compose 文档会给出当前版本对应的 `milvus-standalone-docker-compose.yml` 下载地址。建议使用官方页面中的最新版命令。

示例命令：

```bash
wget https://github.com/milvus-io/milvus/releases/download/v2.5.27/milvus-standalone-docker-compose.yml -O docker-compose.yml
docker compose up -d
docker compose ps
```

Python 连接 Docker 服务：

```python
from pymilvus import MilvusClient

client = MilvusClient(uri="http://localhost:19530", token="root:Milvus")
```

> 注意：不要在已有数据目录上随意执行清空 volume、删除集合、重置服务等命令。学习时建议单独创建一个目录放 docker-compose 文件和数据卷。

## 3. 选择连接目标

下面统一用 `MILVUS_URI` 控制连接方式：

- 本地文件：`./docs_milvus_lite.db`
- Docker 服务：`http://localhost:19530`

第一次学习建议先用 Milvus Lite。

In [1]:
from datetime import datetime
from pathlib import Path

from pymilvus import MilvusClient

# 方式 A：Milvus Lite，本地文件数据库。
MILVUS_URI = "./docs_milvus_lite.db"

# 方式 B：Docker Standalone。本地服务启动后，把上一行替换为：
# MILVUS_URI = "http://localhost:19530"

client = MilvusClient(uri=MILVUS_URI)
collection_name = f"learnone_milvus_demo_{datetime.now():%Y%m%d_%H%M%S}"

print("Milvus URI:", MILVUS_URI)
print("Collection:", collection_name)

ModuleNotFoundError: No module named 'pymilvus'

## 4. 使用 PyMilvus 创建集合

Milvus 中的 collection 类似关系型数据库中的表，用来存储向量字段和标量字段。

这里使用 8 维手写向量，方便离线运行。真实项目中向量维度应和 embedding 模型输出维度一致，例如 OpenAI `text-embedding-3-small` 默认输出维度常见为 1536，`text-embedding-3-large` 常见为 3072。

In [ ]:
dimension = 8

client.create_collection(
    collection_name=collection_name,
    dimension=dimension,
    metric_type="COSINE",
)

print("created:", collection_name)

## 5. 插入示例数据

Milvus 的插入数据通常是一个字典列表。每条记录包含：

- 主键：这里用 `id`
- 向量字段：这里用 `vector`
- 原文或业务字段：这里用 `text`、`topic`

下面插入几条中文学习笔记数据。

In [ ]:
rows = [
    {
        "id": 1,
        "vector": [0.92, 0.10, 0.05, 0.03, 0.02, 0.01, 0.04, 0.08],
        "text": "Milvus Lite 适合本地 notebook 原型验证。",
        "topic": "milvus",
    },
    {
        "id": 2,
        "vector": [0.88, 0.12, 0.04, 0.02, 0.03, 0.02, 0.05, 0.09],
        "text": "Milvus Standalone 可以通过 Docker Compose 在本地启动。",
        "topic": "milvus",
    },
    {
        "id": 3,
        "vector": [0.08, 0.91, 0.05, 0.06, 0.04, 0.02, 0.03, 0.01],
        "text": "LangChain 的 Retriever 可以把向量库接入 RAG 链。",
        "topic": "langchain",
    },
    {
        "id": 4,
        "vector": [0.04, 0.86, 0.07, 0.05, 0.06, 0.03, 0.02, 0.02],
        "text": "Embedding 模型把文本转换成向量，供相似度检索使用。",
        "topic": "embedding",
    },
]

insert_result = client.insert(collection_name=collection_name, data=rows)
insert_result

## 6. 向量搜索与条件过滤

搜索时传入查询向量，Milvus 返回最相似的记录。

常用参数：

- `data`：查询向量列表。
- `limit`：返回条数。
- `output_fields`：返回哪些业务字段。
- `filter`：标量字段过滤表达式。

In [ ]:
query_vector = [[0.90, 0.11, 0.05, 0.03, 0.02, 0.01, 0.04, 0.08]]

results = client.search(
    collection_name=collection_name,
    data=query_vector,
    limit=3,
    output_fields=["text", "topic"],
)

for hit in results[0]:
    print(hit)

In [ ]:
filtered_results = client.search(
    collection_name=collection_name,
    data=query_vector,
    limit=3,
    filter='topic == "milvus"',
    output_fields=["text", "topic"],
)

for hit in filtered_results[0]:
    print(hit)

## 7. 查询标量字段

`query` 更像传统数据库的条件查询，不需要传入向量。适合按主键、标签、租户、来源文件等元数据查记录。

In [ ]:
client.query(
    collection_name=collection_name,
    filter='topic == "langchain"',
    output_fields=["id", "text", "topic"],
)

## 8. 连接和管理工具

### Milvus WebUI

Docker Standalone 启动后，官方文档中默认可以访问：

```text
http://127.0.0.1:9091/webui/
```

WebUI 可以查看本地 Milvus 实例状态、集合、索引、加载状态等。

### Attu

Attu 是 Milvus 常用的图形化管理工具。典型连接信息：

```text
Milvus Address: localhost:19530
Token: root:Milvus
```

如果使用 Docker 启动 Attu，可以参考 Attu 官方仓库或 Milvus 文档中的工具说明。学习时建议只查看集合和数据，不要执行删除、清空、重建等操作。

### Python 检查连接

最简单的检查方式是列出 collections。

In [ ]:
client.list_collections()

## 9. 使用真实 Embedding 模型

前面的 8 维向量是为了离线演示。真实 RAG 项目一般流程是：

1. 文本切分成 chunks。
2. Embedding 模型把 chunks 转成向量。
3. Milvus 存储向量、文本和元数据。
4. 用户问题转向量后进行相似度检索。
5. 把检索结果拼入 prompt，交给 LLM 回答。

下面先用 `pymilvus[model]` 里的默认 embedding 函数演示，不依赖 OpenAI API Key。首次运行可能会下载模型。

In [ ]:
# 首次运行可能需要下载模型依赖。
from pymilvus import model

embedding_fn = model.DefaultEmbeddingFunction()

docs = [
    "Milvus Lite 可以把向量数据库嵌入到 Python 应用中。",
    "Docker Standalone 更接近服务端部署，适合更大的数据量。",
    "LangChain 可以把 Milvus 包装成 VectorStore 和 Retriever。",
]

vectors = embedding_fn.encode_documents(docs)
print("doc count:", len(docs))
print("vector dimension:", len(vectors[0]))

## 10. LangChain 连接 Milvus

LangChain 当前推荐单独安装并导入 `langchain-milvus`：

```python
from langchain_milvus import Milvus
```

连接参数分两种：

```python
# Milvus Lite
connection_args={"uri": "./milvus_langchain.db"}

# Milvus Docker Standalone
connection_args={"uri": "http://localhost:19530", "token": "root:Milvus"}
```

下面示例使用 OpenAI Embeddings。运行前需要在 `.env` 或系统环境变量中配置 `OPENAI_API_KEY`。如果你的项目使用其他兼容 OpenAI 的服务，可以按对应 SDK 配置 `base_url`、`api_key` 和模型名。

In [ ]:
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_milvus import Milvus
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

LANGCHAIN_MILVUS_URI = "./docs_langchain_milvus.db"
# 如果连接 Docker Standalone：
# LANGCHAIN_MILVUS_URI = "http://localhost:19530"

langchain_collection = f"learnone_langchain_demo_{datetime.now():%Y%m%d_%H%M%S}"

documents = [
    Document(
        page_content="Milvus 是开源向量数据库，适合语义搜索和 RAG。",
        metadata={"source": "note", "topic": "milvus"},
    ),
    Document(
        page_content="LangChain 的 VectorStore 抽象可以统一不同向量数据库的使用方式。",
        metadata={"source": "note", "topic": "langchain"},
    ),
    Document(
        page_content="Retriever 会根据用户问题从向量库中取回相关上下文。",
        metadata={"source": "note", "topic": "rag"},
    ),
]

vector_store = Milvus.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=langchain_collection,
    connection_args={"uri": LANGCHAIN_MILVUS_URI},
    index_params={"index_type": "FLAT", "metric_type": "COSINE"},
)

print("LangChain collection:", langchain_collection)

### 10.1 相似度搜索

In [ ]:
hits = vector_store.similarity_search("Milvus 可以用来做什么？", k=2)

for doc in hits:
    print(doc.page_content)
    print(doc.metadata)
    print("---")

### 10.2 带分数的相似度搜索

In [ ]:
hits_with_score = vector_store.similarity_search_with_score("RAG 怎么取回上下文？", k=2)

for doc, score in hits_with_score:
    print("score:", score)
    print(doc.page_content)
    print(doc.metadata)
    print("---")

### 10.3 转成 Retriever

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
retrieved_docs = retriever.invoke("LangChain 如何连接 Milvus？")

for doc in retrieved_docs:
    print(doc.page_content)
    print(doc.metadata)
    print("---")

## 11. LangChain RAG 最小示例

这个示例把 Milvus Retriever 接到一个 LCEL 链中：

```text
question -> retriever -> context -> prompt -> chat model -> answer
```

运行前需要有效的 `OPENAI_API_KEY`。

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是一个严谨的 RAG 助手。只根据给定上下文回答；如果上下文没有答案，就说不知道。\n\n上下文：\n{context}",
        ),
        ("human", "{question}"),
    ]
)

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("Milvus 和 LangChain 在 RAG 中分别负责什么？")

## 12. 常见排查

| 问题 | 可能原因 | 处理方式 |
| --- | --- | --- |
| `ModuleNotFoundError: pymilvus` | 没装 SDK | `pip install -U pymilvus` |
| `ModuleNotFoundError: langchain_milvus` | 没装 LangChain 集成包 | `pip install -U langchain-milvus` |
| Docker 连接失败 | Milvus 服务未启动或端口不对 | `docker compose ps`，确认 `19530` 暴露 |
| WebUI 打不开 | Docker Standalone 未启动或端口冲突 | 访问 `http://127.0.0.1:9091/webui/`，检查容器日志 |
| 向量维度不匹配 | collection 维度和 embedding 输出维度不同 | 新建一个维度正确的新 collection |
| LangChain 检索结果为空 | 数据没插入、集合名不一致、搜索参数不合适 | 打印 collection 名，先用 `similarity_search` 小样本验证 |
| OpenAI 报认证错误 | 环境变量缺失 | 配置 `OPENAI_API_KEY` 并重新启动 notebook kernel |

生产建议：

- 小型原型：Milvus Lite 足够简单。
- 大数据量或多人服务：使用 Docker / Kubernetes 部署 Milvus。
- 不同业务数据使用不同 collection，或使用 metadata 字段做租户、来源、权限过滤。
- 不同 embedding 模型的向量维度可能不同，切换模型时通常需要新建 collection。
- 对已有生产数据，不要在 notebook 中直接执行删除集合、清空数据、重建索引等操作。

## 13. Milvus 2.6+ 版本选择

截图里的课程主题强调 Milvus 2.6+，可以从版本和部署形态两个角度理解。

### 13.1 常见版本形态

| 形态 | 说明 | 适合场景 |
| --- | --- | --- |
| Milvus Lite | 嵌入式本地文件版，随 `pymilvus` 使用 | Notebook、学习、原型验证 |
| Milvus Standalone | 单机服务版，通常用 Docker 启动 | 本地开发、小规模服务验证 |
| Milvus Distributed | 分布式集群版，多个组件独立扩缩容 | 生产、大数据量、高并发 |
| Zilliz Cloud | 基于 Milvus 的商业在线云服务 | 不想自运维、需要托管服务 |

### 13.2 为什么课程常选择 2.6.x

Milvus 2.6.x 更适合学习新 RAG 能力，主要原因是：

- 架构和文档更贴近当前生产推荐形态。
- 对全文搜索、混合检索、稀疏向量、多向量字段等 RAG 需求支持更完整。
- Function、Embedding、Reranker 等能力更方便和上层应用集成。
- 索引、加载、查询和资源管理能力持续增强。

学习时建议先用 Milvus Lite 熟悉 API，再用 Standalone 观察 WebUI / Attu，最后再理解 Distributed 架构。

## 14. Milvus 架构速记

Milvus 架构可以按访问层、协调层、工作节点和存储层理解。

```text
Client SDK
-> Proxy
-> Coordinator
-> Worker Nodes
-> Durable Storage
```

| 层级 | 组件 | 作用 |
| --- | --- | --- |
| 访问层 | Proxy | 接收 SDK 请求，做路由、校验、转发 |
| 协调层 | Coordinator | 管理元数据、任务调度、负载和组件协作 |
| 工作节点 | Streaming Node | 处理写入流、订阅消息、增量数据 |
| 工作节点 | Query Node | 加载 segment，执行向量检索和标量过滤 |
| 工作节点 | Data Node | 处理数据落盘、flush、compaction 等 |
| 存储层 | Meta Storage | 保存 collection、schema、segment 等元数据，常见为 etcd |
| 存储层 | WAL | 写前日志，保证写入可靠性，可基于 Pulsar、Kafka、Woodpecker 等 |
| 存储层 | Object Storage | 保存 binlog、索引文件、segment 数据，常见为 MinIO、S3、Azure Blob 等 |

RAG 项目里最常接触的是 Proxy 暴露的 API、collection schema、index、load 和 search。底层组件理解到“谁负责写、谁负责查、谁负责存”即可。

## 15. Milvus 数据处理流程

RAG 中 Milvus 的数据流可以拆成三段。

### 15.1 数据插入

```text
文本块 + 向量 + 元数据
-> insert / upsert
-> WAL
-> Streaming / Data Node
-> Object Storage
-> Segment
```

插入时要保证：

- 向量维度和 schema 一致。
- 主键策略清楚，是否自动生成 id。
- 元数据字段足够支持过滤，例如 `source`、`page`、`tenant_id`、`doc_type`。

### 15.2 索引建立

```text
Segment 数据
-> Index build
-> 索引文件
-> Object Storage
```

索引用于提升搜索性能。数据量很小时 FLAT 足够；数据量变大后常考虑 HNSW、IVF、AUTOINDEX 等。

### 15.3 数据查询

```text
用户问题向量
-> search
-> Query Node 加载 segment / index
-> 向量相似度计算 + 标量过滤
-> 返回 top_k 文本块和分数
```

查询前要关注 collection 是否已经 load。没有 load 的 collection 可能无法搜索，或者首次搜索会有加载开销。

## 16. Schema 和字段类型

Milvus collection 类似数据库表，schema 决定每条记录有哪些字段。

常见字段：

| 字段 | 作用 | 示例 |
| --- | --- | --- |
| 主键字段 | 唯一标识一条记录 | `id` |
| 向量字段 | 保存 embedding | `dense_vector`、`sparse_vector` |
| 文本字段 | 保存原始 chunk | `text` |
| 字符串字段 | 来源、标题、标签 | `source`、`title` |
| 数字字段 | 页码、时间戳、排序分 | `page`、`created_at` |
| JSON 字段 | 灵活元数据 | `metadata` |
| 数组字段 | 多标签、多分类 | `tags` |
| GEOMETRY 字段 | 空间位置数据 | 地理检索场景 |

向量字段常见类型：

| 类型 | 用途 |
| --- | --- |
| 稠密向量 | 语义检索主力，RAG 最常见 |
| 稀疏向量 | 关键词和 BM25 类检索增强 |
| 二进制向量 | 特定高性能或低存储场景 |

RAG 建 schema 时，不要只存向量。至少要存文本和来源，否则检索到结果后无法组装上下文，也无法追溯答案依据。

## 17. Knowhere、索引和相似度

Knowhere 是 Milvus 底层向量索引和搜索引擎相关组件，整合了多种近似最近邻搜索能力。课程截图里出现的 FAISS、ANNOY、HNSW 等，可以理解为向量检索算法或索引生态。

常见索引速记：

| 索引 | 特点 | 适合场景 |
| --- | --- | --- |
| FLAT | 暴力搜索，召回高，速度随数据量下降 | 小数据、验证准确性 |
| IVF_FLAT | 聚类后搜索，速度和召回可调 | 中等规模 |
| IVF_SQ8 | IVF + 量化，节省内存 | 对精度容忍度较高 |
| HNSW | 图索引，查询快，召回好，内存较高 | RAG 常用高性能检索 |
| DISKANN | 面向大规模磁盘索引 | 超大数据量 |
| AUTOINDEX | 自动选择索引策略 | 希望减少手动调参 |

常见相似度指标：

| 指标 | 含义 | 说明 |
| --- | --- | --- |
| COSINE | 余弦相似度 | 文本 embedding 常用 |
| L2 | 欧氏距离 | 距离越小越相似 |
| IP | 内积 | 部分归一化向量场景常用 |

切换 embedding 模型时，要同时确认向量维度、相似度指标和索引配置是否匹配。

## 18. RAG 检索方式和优化

Milvus 在 RAG 中不只做普通 top_k 向量搜索，还可以组合过滤、范围、分组、全文和混合搜索。

| 检索方式 | 说明 | RAG 用法 |
| --- | --- | --- |
| 基本向量搜索 | 按问题向量返回最相似结果 | 普通知识库问答 |
| 过滤搜索 | 向量搜索 + 标量条件 | 按租户、文档、权限过滤 |
| 范围搜索 | 只返回分数达标结果 | 设置相似度水位线 |
| 分组搜索 | 按字段分组返回 | 避免同一文档刷屏 |
| 主键搜索 | 按 id 查询 | 调试来源、查看原片段 |
| 多向量混合搜索 | 多个向量字段联合检索 | 标题向量 + 正文向量 |
| 全文搜索 | 关键词倒排检索 | 编号、术语、短语命中 |
| 文本匹配 | 标量字符串过滤 | 标签、分类、状态过滤 |

RAG 常见优化链路：

```text
query rewrite
-> hybrid search / dense search
-> metadata filter
-> rerank
-> score threshold
-> context compression
-> LLM answer with sources
```

如果面试里问“如何设置水位线”，可以回答：水位线是相似度或距离阈值，用来过滤低相关片段。它需要用业务评测集调参，不能只凭感觉；阈值过高会召回不足，过低会引入噪声。

## 19. Milvus Full Text Search、BM25 Function 与稀疏向量

Milvus 当前文档中，全文搜索常和 BM25 Function、稀疏向量字段一起出现。它的工程意义是：不只靠 dense embedding 做语义检索，也能保留关键词、型号、编号、人名等精确匹配能力。

典型 schema 思路：

```python
schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True, auto_id=True)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=8192, enable_analyzer=True)
schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024)

bm25_function = Function(
    name="text_bm25_emb",
    input_field_names=["text"],
    output_field_names=["sparse"],
    function_type=FunctionType.BM25,
)
schema.add_function(bm25_function)
```

索引通常是：

```python
index_params.add_index(
    field_name="sparse",
    index_name="sparse_inverted_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
)

index_params.add_index(
    field_name="dense",
    index_name="dense_vector_index",
    index_type="AUTOINDEX",
    metric_type="COSINE",
)
```

关键点：

- `enable_analyzer=True` 让 Milvus 对 `VARCHAR` 文本做分析处理，用于全文搜索。
- BM25 Function 会从 `text` 字段生成 sparse 表示，写入 `sparse` 字段。
- dense 字段负责语义召回，sparse/BM25 负责关键词召回。
- RAG 项目里建议同时保存 `text`、`source`、`page`、`title`、`tenant_id` 等字段，便于生成和权限过滤。

## 20. Hybrid Search 与多路召回融合

Hybrid Search 的核心是同时发起多路检索，再融合排序。

典型两路：

```text
Dense Search: 语义相似，适合同义改写和概念匹配
Sparse/BM25 Search: 关键词精确，适合型号、编号、专有名词
```

融合方式：

| Ranker | 作用 | 适合场景 |
| --- | --- | --- |
| WeightedRanker | 给不同检索路设置权重 | 知道 dense/sparse 哪个更可靠 |
| RRFRanker | Reciprocal Rank Fusion，按排名融合 | 分数尺度不同、不想手调权重 |

面试回答可以这样说：

```text
我不会只用单一向量检索。企业文档里有很多型号、编号、制度条款和人名，dense embedding 容易漏；所以我会用 Milvus 建 dense 字段和 sparse/BM25 字段，同时发起语义检索和关键词检索，再用 WeightedRanker 或 RRF 融合，最后接 reranker 精排。
```

参数调优建议：

- dense 召回不足：增加 query rewrite、提高 dense top_k、换更合适 embedding。
- 关键词漏召回：检查 analyzer、BM25 字段、文本是否保留原始术语。
- 融合后噪声多：增加 metadata filter、score threshold、reranker。
- 延迟高：减少每路 top_k、缓存热门查询、离线预处理上下文。

## 21. Milvus 与 Reranker 的位置

Milvus 负责召回和初排，reranker 负责更精细的相关性判断。

```text
Milvus dense search top 50
+ Milvus BM25 search top 50
-> WeightedRanker / RRFRanker 融合 top 50
-> BGE/Qwen/Cohere reranker 精排 top 5
-> LLM 根据 top 5 回答
```

为什么不直接让 reranker 搜全库？

- reranker 通常是 cross-encoder，需要把 query 和每个候选片段一起输入模型。
- 对全库跑 reranker 成本太高、延迟太高。
- 所以先用向量库快速召回，再对少量候选精排。

Reranker 排查重点：

- 如果正确答案不在召回候选里，reranker 救不了。
- 如果召回候选里有答案但排序靠后，reranker 很有用。
- reranker 输入不要太长，超长 chunk 会被截断。
- 评估时要分别看 rerank 前后的 MRR、NDCG、Recall@K。